In [117]:
import random 
from collections import Counter 
from pathlib import Path 
from rdkit import Chem 
from rdkit.Chem import rdFingerprintGenerator
from rdkit import DataStructs
from rdkit.ML.Cluster import Butina
import pandas as pd 
import useful_rdkit_utils as uru
import sys
import matplotlib.pyplot as plt
import os

In [127]:
sys.path.insert(1, '/Users/paulakramer/Documents/PHD/KinFragLib/kinase_active_learning/notebooks/recombination')
from recombination import cli 
from recombination import recombine_ligands
import utils 
import docking_utils 

In [83]:
HERE = Path(_dh[-1])
PATH_TO_LIB = HERE / '..' / 'data' / 'fragment_library_custom_filtered'
POCKETS = ['AP', 'FP', 'SE', 'GA']

In [9]:
fragment_library = utils.read_fragment_library(PATH_TO_LIB)
fragment_library.keys()

dict_keys(['AP', 'FP', 'SE', 'GA', 'B1', 'B2'])

### Fragment Clustering 

In [85]:
def calculate_fingerprints(pocket): 
    compounds = []
    for ind, romol, smiles in fragment_library[pocket][["ROMol", "smiles"]].itertuples():
        compounds.append((romol, smiles, ind))
    rdkit_gen = rdFingerprintGenerator.GetRDKitFPGenerator(maxPath=5)
    fingerprints = [rdkit_gen.GetFingerprint(mol) for mol, _, _ in compounds]
    #print("Number of compounds converted:", len(fingerprints))
    #print("Fingerprint length per compound:", len(fingerprints[0]))
    return compounds, fingerprints

In [17]:
def tanimoto_distance_matrix(fp_list):
    """Calculate distance matrix for fingerprint list"""
    dissimilarity_matrix = []
    # Notice how we are deliberately skipping the first and last items in the list
    # because we don't need to compare them against themselves
    for i in range(1, len(fp_list)):
        # Compare the current fingerprint against all the previous ones in the list
        similarities = DataStructs.BulkTanimotoSimilarity(fp_list[i], fp_list[:i])
        # Since we need a distance matrix, calculate 1-x for every element in similarity matrix
        dissimilarity_matrix.extend([1 - x for x in similarities])
    return dissimilarity_matrix

In [19]:
def cluster_fingerprints(fingerprints, cutoff=0.2):
    """Cluster fingerprints
    Parameters:
        fingerprints
        cutoff: threshold for the clustering
    """
    # Calculate Tanimoto distance matrix
    distance_matrix = tanimoto_distance_matrix(fingerprints)
    # Now cluster the data with the implemented Butina algorithm:
    clusters = Butina.ClusterData(distance_matrix, len(fingerprints), cutoff, isDistData=True)
    clusters = sorted(clusters, key=len, reverse=True)
    return clusters

In [109]:
def select_clusters(clusters, compounds, count): 
    diverse_frags = []
    for i, c in enumerate(clusters): 
        #if i == count: 
        #    return diverse_frags
        diverse_frags.append(compounds[c[0]][2])
    return diverse_frags

In [110]:
# select diverse fragments from each pocket  
pocket_df = {}
for p in POCKETS: 
    # perform Butina clustering 
    compounds, fingerprints = calculate_fingerprints(p)
    clusters = cluster_fingerprints(fingerprints, cutoff=0.5)
    print(f"Number of clusters for pocket {p}: {len(clusters)}")
    # select first fragment from each cluster (excluding singletons)
    diverse_frags = select_clusters(clusters, compounds, 20)
    print(f"Number of compounds selected for pocket {p}: {len(diverse_frags)}")
    pocket_df[p] = fragment_library[p].loc[diverse_frags]
    for ind in pocket_df[p].index: 
        pocket_df[p].loc[ind].ROMol_original.SetProp("Index", str(ind))

Number of clusters for pocket AP: 52
Number of compounds selected for pocket AP: 52
Number of clusters for pocket FP: 49
Number of compounds selected for pocket FP: 49
Number of clusters for pocket SE: 42
Number of compounds selected for pocket SE: 42
Number of clusters for pocket GA: 38
Number of compounds selected for pocket GA: 38


In [111]:
# write diverse fragments to file 
lib_path = HERE / '..' / 'data' / 'diverse_recomb_library'
utils.delete_folder(lib_path)

utils.save_fragment_library_to_sdfs(lib_path, pd.concat(pocket_df))

### Fragment Recombination 

In [124]:
com_path = Path(HERE / '..' / "data" / "recombination_library")
frag_size = [2,3,4]
for f in frag_size:
    cli.process(com_path, lib_path, 'AP', f)
    # writes recombined ligands to combinatorial_library.sdf 
    recombine_ligands.process(fragment_library, com_path)
    file_name = "combinatorial_library_" + str(f) + ".sdf"
    os.rename(com_path / 'combinatorial_library.sdf', com_path / file_name)
#mols = utils.read_sdf(com_path / "combinatorial_library.sdf")

Subpocket B1 not available.
Subpocket B2 not available.
Number of fragments:  181
Write 2596 results to file 0
Number of resulting ligands:  2596
Number of ligands including duplicates:  2637
Overall number of fragments in queue:  53
Time:  511038.2511458397


[13:30:32] Initializing MetalDisconnector
[13:30:32] Running MetalDisconnector
[13:30:32] Initializing Normalizer
[13:30:32] Running Normalizer
[13:30:32] Running Uncharger
[13:30:32] Initializing MetalDisconnector
[13:30:32] Running MetalDisconnector
[13:30:32] Initializing Normalizer
[13:30:32] Running Normalizer
[13:30:32] Running Uncharger
[13:30:32] Initializing MetalDisconnector
[13:30:32] Running MetalDisconnector
[13:30:32] Initializing Normalizer
[13:30:32] Running Normalizer
[13:30:32] Running Uncharger
[13:30:32] Initializing MetalDisconnector
[13:30:32] Running MetalDisconnector
[13:30:32] Initializing Normalizer
[13:30:32] Running Normalizer
[13:30:32] Running Uncharger
[13:30:32] Initializing MetalDisconnector
[13:30:32] Running MetalDisconnector
[13:30:32] Initializing Normalizer
[13:30:32] Running Normalizer
[13:30:32] Running Uncharger
[13:30:32] Initializing MetalDisconnector
[13:30:32] Running MetalDisconnector
[13:30:32] Initializing Normalizer
[13:30:32] Running No

Number of recombined ligands from all iterations: 2596
Subpocket B1 not available.
Subpocket B2 not available.
Number of fragments:  181
Write 24651 results to file 0
Number of resulting ligands:  24651
Number of ligands including duplicates:  47998
Overall number of fragments in queue:  1450
Time:  511043.2659089565


[13:30:37] Initializing MetalDisconnector
[13:30:37] Running MetalDisconnector
[13:30:37] Initializing Normalizer
[13:30:37] Running Normalizer
[13:30:37] Running Uncharger
[13:30:37] Initializing MetalDisconnector
[13:30:37] Running MetalDisconnector
[13:30:37] Initializing Normalizer
[13:30:37] Running Normalizer
[13:30:37] Running Uncharger
[13:30:37] Initializing MetalDisconnector
[13:30:37] Running MetalDisconnector
[13:30:37] Initializing Normalizer
[13:30:37] Running Normalizer
[13:30:37] Running Uncharger
[13:30:37] Initializing MetalDisconnector
[13:30:37] Running MetalDisconnector
[13:30:37] Initializing Normalizer
[13:30:37] Running Normalizer
[13:30:37] Running Uncharger
[13:30:37] Initializing MetalDisconnector
[13:30:37] Running MetalDisconnector
[13:30:37] Initializing Normalizer
[13:30:37] Running Normalizer
[13:30:37] Running Uncharger
[13:30:37] Initializing MetalDisconnector
[13:30:37] Running MetalDisconnector
[13:30:37] Initializing Normalizer
[13:30:37] Running No

Number of recombined ligands from all iterations: 24651
Subpocket B1 not available.
Subpocket B2 not available.
Number of fragments:  181
Write 35636 results to file 0
Number of resulting ligands:  35636
Number of ligands including duplicates:  61620
Overall number of fragments in queue:  6905
Time:  511097.13738679886


[13:31:31] Initializing MetalDisconnector
[13:31:31] Running MetalDisconnector
[13:31:31] Initializing Normalizer
[13:31:31] Running Normalizer
[13:31:31] Running Uncharger
[13:31:31] Initializing MetalDisconnector
[13:31:31] Running MetalDisconnector
[13:31:31] Initializing Normalizer
[13:31:31] Running Normalizer
[13:31:31] Running Uncharger
[13:31:31] Initializing MetalDisconnector
[13:31:31] Running MetalDisconnector
[13:31:31] Initializing Normalizer
[13:31:31] Running Normalizer
[13:31:31] Running Uncharger
[13:31:31] Initializing MetalDisconnector
[13:31:31] Running MetalDisconnector
[13:31:31] Initializing Normalizer
[13:31:31] Running Normalizer
[13:31:31] Running Uncharger
[13:31:31] Initializing MetalDisconnector
[13:31:31] Running MetalDisconnector
[13:31:31] Initializing Normalizer
[13:31:31] Running Normalizer
[13:31:31] Running Uncharger
[13:31:31] Initializing MetalDisconnector
[13:31:31] Running MetalDisconnector
[13:31:31] Initializing Normalizer
[13:31:31] Running No

Number of recombined ligands from all iterations: 35636


In [126]:
with Chem.SDWriter(str(com_path / "combinatorial_library.sdf")) as w:
    for f in frag_size:
        file_name = "combinatorial_library_" + str(f) + ".sdf"
        with Chem.SDMolSupplier(str(com_path / file_name), removeHs=False) as mol_file:
            for m in mol_file:
                w.write(m)

In [116]:
# plot cluster size
#fig, ax = plt.subplots(figsize=(15, 4))
#ax.set_xlabel("Cluster index")
#ax.set_ylabel("Number of molecules")
#ax.bar(range(1, len(clusters) + 1), [len(c) for c in clusters], lw=5);

### Template Docking Preparation 

In [133]:
# extract AP fragments 
mols_docking = []
core_frag = []
mols = [mol for mol in Chem.SDMolSupplier(str(com_path / "combinatorial_library.sdf"))]
print(f"Reading {len(mols)}")
for mol in mols: 
    mol = docking_utils.compute_coordinates(mol)
    mol_ap = docking_utils.extract_pocket(mol, "AP")
    if mol_ap is None: 
        continue
    core_frag.append(mol_ap)
    mols_docking.append(mol)
print(f"Extracted AP fragments")
ligand_files, template_files = docking_utils.create_template_docking_files(mols_docking, core_frag, com_path / "temp" / "docking_input", fragment_library)
print(f"Extracted {len(template_files)} different core fragments")

Reading 62883
Extracted AP fragments
Extracted 52 different core fragments


In [134]:
len(ligand_files)

52

### Docking

In [135]:
PATH_HYDE = Path('/Applications/Hydescorer.app/Contents/MacOS/Hydescorer')
PATH_FLEXX = Path('/Applications/FlexX.app/Contents/MacOS/FlexX')
DOCKING_CONFIG = HERE / '..' / 'data' / 'docking_config' / '5n1f'

In [ ]:
print(ligand_files)
concat_docked = docking_utils.full_docking(ligand_files, template_files, com_path / "temp" / "docking_output", DOCKING_CONFIG, PATH_FLEXX, PATH_HYDE)
print(f"done docking, concat_docking len: {len(concat_docked)}")  
final_docked = docking_utils.filter_poses_id(mols_docking, concat_docked)
print(f"Successful dockings: {len([f for f in final_docked if f is not None])} / {len(mols_docking)}")
